## Задание 5

Напишите функцию `function01`. Она должна иметь следующую сигнатуру:

`def function01(tensor: torch.Tensor, count_over: str):`

Если `count_over` равен `columns`, верните среднее тензора по колонкам. Если равен `rows`, то верните среднее по рядам. Гарантируется, что тензор будет матрицей (то есть будет иметь размерность 2).


In [ ]:
import torch


def function01(tensor: torch.Tensor, count_over: str):
    if count_over == "columns":
        return tensor.mean(dim=0)  
    if count_over == "rows":
        return tensor.mean(dim=1) 

In [2]:
x = torch.arange(15, dtype=torch.float32).reshape(5, 3)

x

tensor([[ 0.,  1.,  2.],
        [ 3.,  4.,  5.],
        [ 6.,  7.,  8.],
        [ 9., 10., 11.],
        [12., 13., 14.]])

In [3]:
function01(x, 'columns')

tensor([6., 7., 8.])

In [4]:
function01(x, 'rows')

tensor([ 1.,  4.,  7., 10., 13.])

## Задание 6

Напишите функцию `function02`. Функции на вход должен приходить датасет — тензор-матрица признаков объектов. Ваша функция должна создать тензор-вектор с весами (пусть они будут из равномерного распределения на отрезке от 0 до 1) и вернуть их для дальнейшего обучения линейной регрессии без свободного коэффициента. Сделайте эти веса типа `float32`, для них нужно будет в процессе обучения вычислять градиенты (воспользуйтесь `requires_grad`).


In [5]:
import torch


def function02(x: torch.Tensor):
    return torch.rand(x.shape[1], dtype=torch.float32, requires_grad=True)

In [6]:
x = torch.rand(7, 5)

x

tensor([[0.2757, 0.2043, 0.1700, 0.9087, 0.2743],
        [0.2410, 0.7089, 0.0777, 0.1966, 0.4431],
        [0.4491, 0.4141, 0.5535, 0.8035, 0.4169],
        [0.3889, 0.1352, 0.3165, 0.6763, 0.7586],
        [0.7434, 0.2942, 0.9308, 0.3763, 0.7160],
        [0.8723, 0.0250, 0.2532, 0.7584, 0.0690],
        [0.6419, 0.6918, 0.7235, 0.1620, 0.8213]])

In [7]:
w = function02(x)

w

tensor([0.6691, 0.2770, 0.0339, 0.2965, 0.4093], requires_grad=True)

## Задание 7

Напишите функцию `function03`. Она должна принимать тензор-матрицу с объектами и тензор с правильными ответами, будем решать задачу регрессии: 

`def function03(x: torch.Tensor, y: torch.Tensor):`

Создайте внутри функции веса для линейной регрессии (без свободного коэффициента), можете воспользоваться функцией из предыдущего степа, с помощью градиентного спуска подберите оптимальные веса для входных данных (используйте длину шага около `1e-2`). Верните эти веса из функции. Ваши обученные веса должны давать **MSE** на обучающей выборке меньше единицы.

Входные данные будут выглядеть примерно так:

```
n_features = 2
n_objects = 300

w_true = torch.randn(n_features)
X = (torch.rand(n_objects, n_features) - 0.5) * 5
Y = X @ w_true + torch.randn(n_objects) / 2
```



In [ ]:
import torch


def function03(x: torch.Tensor, y: torch.Tensor):
    w = torch.rand(x.shape[1], dtype=torch.float32, requires_grad=True)  # инициализируем веса как в прошлом задании

    mse = torch.tensor([100])  

    while mse.item() > 1: 
        y_pred = x @ w  

        mse = torch.mean((y_pred - y) ** 2)  # считаем ошибку

        mse.backward()  

        with torch.no_grad():  
            w = w - w.grad * 1e-2  
        w.requires_grad = True 

    return w  


In [9]:
n_features = 2
n_objects = 300

w_true = torch.randn(n_features)
X = (torch.rand(n_objects, n_features) - 0.5) * 5
Y = X @ w_true + torch.randn(n_objects) / 2

w = function03(X, Y)

(X @ w - Y).square().mean()

tensor(0.8862, grad_fn=<MeanBackward0>)

## Задание 8

Напишите функцию `function04`. Она должна принимать тензор-матрицу с объектами и тензор с правильными ответами, будем решать задачу регрессии: 

`def function04(x: torch.Tensor, y: torch.Tensor):`

Создайте внутри функции полносвязный слой, обучите этот полносвязный слой на входных данных с помощью градиентного спуска (используйте длину шага около `1e-2`). Верните его из функции. Ваш полносвязный слой должен давать **MSE** на обучающей выборке меньше 0.3.

Входные данные будут выглядеть примерно так:

```
n_features = 2
n_objects = 300

w_true = torch.randn(n_features)
X = (torch.rand(n_objects, n_features) - 0.5) * 5
Y = X @ w_true + torch.randn(n_objects) / 2
```



In [11]:
import torch
import torch.nn as nn


def function04(x: torch.Tensor, y: torch.Tensor):
    layer = nn.Linear(x.shape[1], 1, bias=True)

    mse = torch.tensor([100])

    while mse > 0.3:
        mse = torch.mean((layer(x).ravel() - y) ** 2)

        mse.backward()

        with torch.no_grad():
            layer.weight -= layer.weight.grad * 1e-2
            layer.bias -= layer.bias.grad * 1e-2

        layer.zero_grad()

    return layer

In [12]:
n_features = 2
n_objects = 300

w_true = torch.randn(n_features)
X = (torch.rand(n_objects, n_features) - 0.5) * 5
Y = X @ w_true + torch.randn(n_objects) / 2

layer = function04(X, Y)

In [13]:
(layer(X).ravel() - Y).square().mean()

tensor(0.2927, grad_fn=<MeanBackward0>)